In [8]:
import json 
import os  

with open('config.json', 'r') as f:
    data = json.load(f)

pathway_gen = os.path.abspath(data["python_files"])
pathway_temp = os.path.abspath(data["publications"])
pathway = os.path.join(pathway_temp, "tennie2010evidence")
original_data_pathway = os.path.join(pathway, "original_data")

complete_path_1 = os.path.join(original_data_pathway, "Tennie_Call_Tomasello_2010_float peanut_raw data.csv")

out_pathway = os.path.join(pathway, "standardized_data")
if not os.path.exists(out_pathway):
    os.makedirs(out_pathway)

In [9]:
import pandas as pd
import numpy as np
import pyreadstat

df = pd.read_csv(complete_path_1)


df['study_id']="tennie2010evidence"
df.columns = map(str.lower, df.columns)
df=df.applymap(lambda s: s.lower() if type(s) == str else s)
# df.columns


In [10]:
df.rename(columns={"subject": "ape",
    "age at test in years":"age_original",
    "trial length":"trial_length",
    "spit y/n":"spit_y_n",
    "sp number":"sp_number",
    "1st spit time":"1st_spit_time",
    "success y/n":"success_y_n",
    "succ time":"succ_time",
    "drinker y/n":"drinker_y_n",
    "1st dr time":"1st_dr_time",
    "dr number":"dr_number"}, inplace=True)

In [11]:
comp_path_name_errors = os.path.join(pathway_gen, "common_name_errors.csv")

df_name  = pd.read_csv(comp_path_name_errors)
df['ape'] = df['ape'].str.rstrip()
for x,y in zip(df_name['wrong'],df_name['right']):
    df['ape'].replace(x, y, inplace=True)

comp_path_ape_info = os.path.join(pathway_gen, "apes_includeindatabase.csv")
apedf = pd.read_csv(comp_path_ape_info)
df= df.merge(apedf,left_on='ape', right_on='name', how='left')

In [12]:
df.replace('n.a.', np.nan, inplace=True)

In [13]:
# df.columns
df.rename(columns={"ape": "participant", 
                   "trial_length":"trial_length_seconds",
                   'sp_number':"spit_number",
                   'succ_time':"success_time",
                   "drinker_y_n":"use_drinker_y_n",
                   "1st_dr_time":"1st_use_drinker_time",
                   "dr_number":"number_times_used_drinker"}, inplace=True)

In [14]:
tennie2010evidence_standardized=df[['study_id', 'participant','age_original', 'sex',  'species','condition', 
                                    'trial',  'trial_length_seconds',
       'spit_y_n', 'spit_number', '1st_spit_time', 'success_y_n', 'success_time',
       'use_drinker_y_n', '1st_use_drinker_time', 'number_times_used_drinker', 'comment']]
comp_out_path_stand = os.path.join(out_pathway, 'tennie2010evidence_standardized.csv')
tennie2010evidence_standardized.to_csv(comp_out_path_stand, encoding='utf-8-sig', index=False)

names =tennie2010evidence_standardized.columns.tolist()
df = pd.DataFrame(names)
df = df.rename(columns={0: "column_name"})
df["description"] = ""
tennie2010evidence_glossary=df[["column_name", "description"]]

comp_out_path_glossary = os.path.join(out_pathway, 'tennie2010evidence_glossary.csv')
tennie2010evidence_glossary.to_csv(comp_out_path_glossary, encoding='utf-8-sig', index=False)

